# COLUMNS TRANSFORMER. PIPELINE

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder, RobustScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.impute import SimpleImputer

from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

from sklearn.compose import ColumnTransformer
#from sklearn.compose import make_column_selector
from sklearn.pipeline import Pipeline


import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

import warnings

warnings.filterwarnings('ignore')

In [ ]:
def categoricas_unicos(dataframe, var_cat, vc_out=True):
    """
    Muestra valores unicos y conteo de las variables categóricas.
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_cat: lista de variables categóricas (cadena)
    - vc_out: booleano para mostrar o no (True/False) el resultado de "value_counts()"
        Por defecto lo muestra

    -------------------------------
    SALIDA:
    No devuelve valor. Saca por pantalla la información

    """

    for discreta in var_cat:

        print(f'Variable {discreta.upper()}:')
        print('Valores unicos: ')
        print(dataframe[discreta].unique(), end='\n'*2)

        if vc_out:
            print(dataframe[discreta].value_counts(), end='\n'*2)





In [ ]:
def graficas_var_categorica(dataframe, var_cat):
    """
    Realiza los diagramas de barras de las variables categóricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_cat: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """


    colores = sns.color_palette("husl", len(var_cat))

    # creacion matriz de graficas
    fig, axes = plt.subplots(len(var_cat), 1, \
                             figsize=(10, 4*len(var_cat)),\
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.4})

    ax = axes.ravel()

    # dibujamos las graficas
    for idx,variable in enumerate(var_cat):

        sns.countplot(dataframe[variable], ax=ax[idx],palette=colores)

        ax[idx].set_title(f'DIAGRAMA DE BARRAS {variable}')
        ax[idx].set_xlabel(f'Valores únicos (categorias) de {variable}')
        ax[idx].set_ylabel("Frequencia")


In [ ]:
def graf_histo_box_numericas(dataframe, var_num):
    """
    Realiza graficas histograma y boxplot de variables numéricas de un dataframe
    -------------------------------
    Parametros de entrada:
    - dataframe: DF de pandas
    - var_num: lista de variables numéricas (cadena)

    -------------------------------
    SALIDA:
    No devuelve valor. Dibuja las gráficas por pantalla

    """

    TABLEAU_CMP = ('tab:blue', 'tab:orange', 'tab:green', 'tab:red', 'tab:purple', 'tab:brown', 'tab:pink', \
                   'tab:gray','tab:olive', 'tab:cyan')

    fig, axes = plt.subplots(len(var_num), 2, \
                             figsize=(20, 5 * len(var_num)), \
                             gridspec_kw={'hspace': 0.4, 'wspace': 0.1})
    ax = axes.ravel()

    # graficas distribucion y boxplot de cada atributo
    for idx, atributo in enumerate(var_num):

        # distribucion (histograma)
        sns.distplot(dataframe[atributo], bins=30, ax=ax[2 * idx], \
                     color=TABLEAU_CMP[idx % len(TABLEAU_CMP)], \
                     hist_kws={'alpha': 0.15})

        # titulo, etiquetas histograma
        ax[2 * idx].set_title(f'HISTOGRAMA {atributo}')
        ax[2 * idx].set_xlabel(f'Valores {atributo}')
        ax[2 * idx].set_ylabel("Frequencia")

        # boxplot
        sns.boxplot(x=atributo, data=dataframe, ax=ax[2 * idx + 1], color=TABLEAU_CMP[idx % len(TABLEAU_CMP)])

        # titulo, etiquetas boxplot
        ax[2 * idx + 1].set_title(f'BOXPLOT {atributo}')
        ax[2 * idx + 1].set_xlabel(f'Valores {atributo}')



## CASO MAS SENCILLO

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df1 = pd.read_csv('/content/drive/MyDrive/ESPECIALISTA IA/adult1.csv')

df1.info()

In [ ]:
df1.isnull().sum()

In [ ]:
target = 'income'

X = df1.drop(columns=target)

y = df1[target]

X.shape, y.shape

In [ ]:
atrib_num = X.select_dtypes(exclude='object').columns.to_list()

atrib_num

In [ ]:
atrib_cat = X.select_dtypes(include='object').columns.to_list()

atrib_cat

**El target lo podemos tratar ya:**

In [ ]:
label_enc = LabelEncoder()

y_enc = label_enc.fit_transform(y)


In [ ]:
y_enc

**En cuanto a los atributos el tratamiento a seguir es convertir a numérico las categóricas (utilizamos como ejemplo OHE) y escalar las numéricas (Standar Scaler). Veamos como lo hacemos con "ColumnnTransformer":**

In [ ]:
preprocessor = ColumnTransformer([
    ('scale', StandardScaler(), atrib_num),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), atrib_cat)],
    remainder='passthrough')

In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape

Respetamos la regla de entrenar con TRAIN:

In [ ]:
Xtrain_prep = preprocessor.fit_transform(Xtrain)
Xtest_prep  = preprocessor.transform(Xtest)

Xtrain_prep.shape, Xtest_prep.shape

Para recuperarlo como DF, hay que tener en cuenta que en el "Columns Transformer" primero tratamos las numéricas, y despues las categóricas. Al utilizar para éstas últimas OHE, nos salen más columnas de las originales y recuperamos sus nombres del método correspondiente del transformador:

In [ ]:
encoded_cat = preprocessor.named_transformers_['onehot'].get_feature_names_out(atrib_cat)

nombre_columnas = np.concatenate([atrib_num, encoded_cat])

nombre_columnas


In [ ]:

df_train_prep = pd.DataFrame(Xtrain_prep, columns=nombre_columnas)

df_train_prep.sample(n=10)


In [ ]:
df_train_prep.describe()

In [ ]:
df_train_prep.workclass_goverment.value_counts()

**REPRESENTACIÓN "GRÁFICA"**

In [ ]:
from sklearn import set_config
set_config(display='diagram')

preprocessor

In [ ]:
set_config(display='text')

preprocessor

In [ ]:
clf = GaussianNB()

clf.fit(Xtrain_prep, ytrain)



In [ ]:
clf.score(Xtrain_prep, ytrain)

In [ ]:
clf.score(Xtest_prep, ytest)

## CASO UN POCO MAS COMPLEJO: PIPELINE

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df2 = pd.read_csv('/content/drive/MyDrive/ESPECIALISTA IA/adult2.csv')

df2.info()

In [ ]:
df2.isnull().sum()

In [ ]:
target = 'income'

X = df2.drop(columns=target)

y = df2[target]

X.shape, y.shape

In [ ]:
atrib_num = X.select_dtypes(exclude='object').columns.to_list()

atrib_num

In [ ]:
atrib_cat = X.select_dtypes(include='object').columns.to_list()

atrib_cat

**El target lo podemos tratar ya:**

In [ ]:
label_enc = LabelEncoder()

y_enc = label_enc.fit_transform(y)


In [ ]:
y_enc

**La novedad es que ahora en las numéricas necesitamos 2 procesados, la imputación de nulos y el escalado. En este caso nos sirve de ayuda "Pipeline" (tratamiento en serie, uno y despues el otro):**

In [ ]:
# Transformaciones para las variables numéricas

numeric_transformer = Pipeline(
                        steps=[
                            ('imputer', SimpleImputer(strategy='median')),
                            ('scaler', StandardScaler())
                        ]
                      )

In [ ]:
set_config(display='diagram')

numeric_transformer

In [ ]:
set_config(display='text')

numeric_transformer

**Para las categóricas no ha cambiado nada. Lo ponemos todo junto:**

In [ ]:
preprocessor = ColumnTransformer(
                    transformers=[
                        ('numeric', numeric_transformer, atrib_num),
                        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), atrib_cat)
                    ],
                    remainder='passthrough')


In [ ]:
Xtrain, Xtest, ytrain, ytest = train_test_split(X, y_enc, test_size=0.2, stratify=y_enc, random_state=42)

Xtrain.shape, Xtest.shape, ytrain.shape, ytest.shape

Respetamos la regla de entrenar con TRAIN:

In [ ]:
Xtrain_prep = preprocessor.fit_transform(Xtrain)
Xtest_prep  = preprocessor.transform(Xtest)

Xtrain_prep.shape, Xtest_prep.shape

In [ ]:
encoded_cat = preprocessor.named_transformers_['onehot'].get_feature_names_out(atrib_cat)
nombre_columnas = np.concatenate([atrib_num, encoded_cat])

nombre_columnas


In [ ]:

df_train_prep = pd.DataFrame(Xtrain_prep, columns=nombre_columnas)

df_train_prep.sample(n=10)


In [ ]:
df_train_prep.isnull().sum()

In [ ]:
df_train_prep.describe()

In [ ]:
df_train_prep.workclass_goverment.value_counts()

In [ ]:
from sklearn import set_config
set_config(display='diagram')

preprocessor

In [ ]:
set_config(display='text')

preprocessor